# Step 1 — Connect Python to SQL Server

In [1]:
import pyodbc
import pandas as pd

In [6]:
# Connection parameters
SERVER   = 'localhost'          # or 'localhost\\SQLEXPRESS'
DATABASE = 'NawyProptechDB'
#USERNAME = 'sa'                 # or your Windows auth
#PASSWORD = 'Password'       

In [13]:
# Connection string
conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    #f"UID={USERNAME};"
    #f"PWD={PASSWORD};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

In [14]:
# Test connection
try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    cursor.execute("SELECT DB_NAME() AS db, SUSER_NAME() AS login_name")
    row = cursor.fetchone()
    print(f" Connected to: {row[0]}")
    print(f" Logged in as: {row[1]}")
    
    # Count tables in nawy schema
    cursor.execute("SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'nawy'")
    count = cursor.fetchone()[0]
    print(f" Tables in nawy schema: {count}")
    
except Exception as e:
    print(f" Connection failed: {e}")

 Connected to: NawyProptechDB
 Logged in as: DESKTOP-A9H2SVR\user
 Tables in nawy schema: 21


# Step 2 — Load Shares Data into DataFrames
Goal: Pull the four tables we need into pandas DataFrames so we can analyze them.

In [16]:
# Step 2: Load Nawy Shares Data into DataFrames

import pandas as pd
import numpy as np

# Reuse the connection from Step 1


# --- Load 1: Offerings with property details ---
df_offerings = pd.read_sql("""
    SELECT
        so.offering_id,
        so.property_id,
        so.total_value,
        so.share_price,
        so.total_shares,
        so.shares_available,
        CAST(so.offering_open_date  AS DATE) AS offering_open_date,
        CAST(so.offering_close_date AS DATE) AS offering_close_date,
        so.status              AS offering_status,
        so.exit_condition,
        p.title                AS property_title,
        p.city,
        p.district,
        p.property_type,
        p.area_sqm,
        p.bedrooms,
        p.finishing_status,
        CAST(p.delivery_date AS DATE) AS delivery_date,
        p.price                AS property_price,
        p.developer_id,
        d.name                 AS developer_name,
        d.partnership_tier
    FROM nawy.SHARE_OFFERING so
    INNER JOIN nawy.PROPERTY p ON so.property_id = p.property_id
    LEFT JOIN nawy.DEVELOPER d ON p.developer_id = d.developer_id
""", conn)


# --- Load 2: Investments ---
df_investments = pd.read_sql("""
    SELECT
        si.investment_id,
        si.offering_id,
        si.user_id,
        si.shares_purchased,
        si.total_amount,
        CAST(si.investment_date AS DATETIME) AS investment_date,
        si.status              AS investment_status,
        u.full_name            AS investor_name,
        u.user_type            AS investor_type
    FROM nawy.SHARE_INVESTMENT si
    INNER JOIN nawy.[USER] u ON si.user_id = u.user_id
""", conn)


# --- Load 3: Exits (fully corrected) ---
df_exits = pd.read_sql("""
    SELECT
        se.exit_id,
        se.investment_id,
        se.exit_price_per_share,
        se.total_sale_value,
        se.exit_fee_percent,
        se.profit_loss,
        CAST(
            se.profit_loss * 100.0 / NULLIF(si.total_amount, 0)
            AS DECIMAL(10,2)
        ) AS roi_pct,
        CAST(se.exit_date AS DATE) AS exit_date,
        se.status              AS exit_status,
        si.offering_id,
        si.user_id,
        si.shares_purchased,
        si.total_amount        AS original_investment,
        CAST(si.investment_date AS DATETIME) AS investment_date
    FROM nawy.SHARE_EXIT se
    INNER JOIN nawy.SHARE_INVESTMENT si ON se.investment_id = si.investment_id
""", conn)


# --- Summary ---
print("-" * 50)
print("DATA LOAD SUMMARY")
print("-" * 50)
print(f"Offerings:   {len(df_offerings):,} rows × {df_offerings.shape[1]} cols")
print(f"Investments: {len(df_investments):,} rows × {df_investments.shape[1]} cols")
print(f"Exits:       {len(df_exits):,} rows × {df_exits.shape[1]} cols")
print()
print("Offering columns:", list(df_offerings.columns))
print()
print("Exit columns:", list(df_exits.columns))

C:\Users\user\AppData\Local\Temp\ipykernel_8080\924952237.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_offerings = pd.read_sql("""
C:\Users\user\AppData\Local\Temp\ipykernel_8080\924952237.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_investments = pd.read_sql("""
C:\Users\user\AppData\Local\Temp\ipykernel_8080\924952237.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_exits = pd.read_sql("""


--------------------------------------------------
DATA LOAD SUMMARY
--------------------------------------------------
Offerings:   9 rows × 22 cols
Investments: 531 rows × 9 cols
Exits:       154 rows × 14 cols

Offering columns: ['offering_id', 'property_id', 'total_value', 'share_price', 'total_shares', 'shares_available', 'offering_open_date', 'offering_close_date', 'offering_status', 'exit_condition', 'property_title', 'city', 'district', 'property_type', 'area_sqm', 'bedrooms', 'finishing_status', 'delivery_date', 'property_price', 'developer_id', 'developer_name', 'partnership_tier']

Exit columns: ['exit_id', 'investment_id', 'exit_price_per_share', 'total_sale_value', 'exit_fee_percent', 'profit_loss', 'roi_pct', 'exit_date', 'exit_status', 'offering_id', 'user_id', 'shares_purchased', 'original_investment', 'investment_date']


# Step 3 — Data Profile & Quality Check
Goal: Before analyzing, we need to know what's actually in the data — how many nulls, what ranges look like, and whether anything is broken.

In [17]:
import pandas as pd
import numpy as np

In [18]:
# --- 3.1: Basic shape ---
print("-" * 50)
print("DATA SHAPES")
print("-" * 50)
print(f"Offerings:   {df_offerings.shape[0]:>5} rows × {df_offerings.shape[1]:>2} cols")
print(f"Investments: {df_investments.shape[0]:>5} rows × {df_investments.shape[1]:>2} cols")
print(f"Exits:       {df_exits.shape[0]:>5} rows × {df_exits.shape[1]:>2} cols")
print()

--------------------------------------------------
DATA SHAPES
--------------------------------------------------
Offerings:       9 rows × 22 cols
Investments:   531 rows ×  9 cols
Exits:         154 rows × 14 cols



In [19]:
# --- 3.2: Missing values per column ---
def null_report(df, name):
    """Print null counts and percentages per column."""
    nulls = df.isnull().sum()
    pct = (nulls / len(df) * 100).round(1)
    report = pd.DataFrame({
        'null_count': nulls,
        'null_pct': pct
    })
    report = report[report['null_count'] > 0].sort_values('null_pct', ascending=False)
    
    print(f"--- {name}: columns with missing values ---")
    if len(report) == 0:
        print("    No missing values\n")
    else:
        print(report.to_string())
        print()

null_report(df_offerings, 'Offerings')
null_report(df_investments, 'Investments')
null_report(df_exits, 'Exits')


--- Offerings: columns with missing values ---
                  null_count  null_pct
delivery_date              8      88.9
developer_id               6      66.7
developer_name             6      66.7
partnership_tier           6      66.7

--- Investments: columns with missing values ---
    No missing values

--- Exits: columns with missing values ---
    No missing values



In [20]:
# --- 3.3: Numeric column statistics ---
print("-" * 50)
print("NUMERIC STATISTICS — OFFERINGS")
print("-" * 50)
print(df_offerings[['total_value', 'share_price', 'total_shares',
                    'shares_available', 'area_sqm', 'bedrooms']].describe().round(2))

print()
print("-" * 50)
print("NUMERIC STATISTICS — INVESTMENTS")
print("-" * 50)
print(df_investments[['shares_purchased', 'total_amount']].describe().round(2))

print()
print("-" * 50)
print("NUMERIC STATISTICS — EXITS")
print("-" * 50)
print(df_exits[['exit_price_per_share', 'total_sale_value',
                'exit_fee_percent', 'profit_loss', 'roi_pct',
                'original_investment']].describe().round(2))

--------------------------------------------------
NUMERIC STATISTICS — OFFERINGS
--------------------------------------------------
       total_value  share_price  total_shares  shares_available  area_sqm  \
count         9.00         9.00           9.0               9.0      9.00   
mean   10678324.89     10678.32        1000.0               0.0    276.22   
std     6665571.73      6665.57           0.0               0.0    130.83   
min     2827440.00      2827.44        1000.0               0.0    102.00   
25%     4309200.00      4309.20        1000.0               0.0    201.00   
50%    10692864.00     10692.86        1000.0               0.0    307.00   
75%    14699160.00     14699.16        1000.0               0.0    323.00   
max    23333520.00     23333.52        1000.0               0.0    528.00   

       bedrooms  
count      9.00  
mean       4.22  
std        1.92  
min        1.00  
25%        3.00  
50%        4.00  
75%        6.00  
max        6.00  

----------

In [21]:
# --- 3.4: Categorical distributions ---
print("-" * 50)
print("CATEGORICAL DISTRIBUTIONS")
print("-" * 50)

print("\nOfferings by status:")
print(df_offerings['offering_status'].value_counts())

print("\nOfferings by city:")
print(df_offerings['city'].value_counts().head(10))

print("\nOfferings by property type:")
print(df_offerings['property_type'].value_counts())

print("\nExits by status:")
print(df_exits['exit_status'].value_counts())

--------------------------------------------------
CATEGORICAL DISTRIBUTIONS
--------------------------------------------------

Offerings by status:
offering_status
sold_out    6
closed      3
Name: count, dtype: int64

Offerings by city:
city
Cairo                         4
New Administrative Capital    2
North Coast                   2
Giza                          1
Name: count, dtype: int64

Offerings by property type:
property_type
villa        4
apartment    3
townhouse    2
Name: count, dtype: int64

Exits by status:
exit_status
completed    154
Name: count, dtype: int64


In [22]:
# --- 3.5: Date ranges ---
print("-" * 50)
print("DATE RANGES")
print("-" * 50)

print(f"Offering open dates:  {df_offerings['offering_open_date'].min()} → {df_offerings['offering_open_date'].max()}")
print(f"Investment dates:     {df_investments['investment_date'].min()} → {df_investments['investment_date'].max()}")
print(f"Exit dates:           {df_exits['exit_date'].min()} → {df_exits['exit_date'].max()}")

--------------------------------------------------
DATE RANGES
--------------------------------------------------
Offering open dates:  2021-03-30 → 2026-06-05
Investment dates:     2022-02-01 00:00:00 → 2022-04-01 00:00:00
Exit dates:           2022-08-13 → 2023-09-18


In [23]:
# --- 3.6: Sanity checks ---
print("-" * 50)
print("SANITY CHECKS")
print("-" * 50)

# Check 1: shares_sold cannot exceed total_shares
df_offerings['shares_sold'] = df_offerings['total_shares'] - df_offerings['shares_available']
over_subscribed = df_offerings[df_offerings['shares_sold'] > df_offerings['total_shares']]
print(f"1. Over-subscribed offerings: {len(over_subscribed)}  (should be 0)")

# Check 2: any negative amounts?
neg_amounts = df_investments[df_investments['total_amount'] <= 0]
print(f"2. Negative/zero investment amounts: {len(neg_amounts)}  (should be 0)")

# Check 3: any negative profit?
neg_profit = df_exits[df_exits['profit_loss'] < 0]
print(f"3. Losses in exits: {len(neg_profit)} of {len(df_exits)}")

# Check 4: any ROI > 500%?
extreme_roi = df_exits[df_exits['roi_pct'] > 500]
print(f"4. Extreme ROIs (>500%): {len(extreme_roi)}  (check for data errors)")

# Check 5: exits with negative holding days (impossible)
df_exits['holding_days'] = (pd.to_datetime(df_exits['exit_date']) - pd.to_datetime(df_exits['investment_date'])).dt.days
neg_days = df_exits[df_exits['holding_days'] < 0]
print(f"5. Exits before investment date: {len(neg_days)}  (should be 0)")

print()

--------------------------------------------------
SANITY CHECKS
--------------------------------------------------
1. Over-subscribed offerings: 0  (should be 0)
2. Negative/zero investment amounts: 0  (should be 0)
3. Losses in exits: 0 of 154
4. Extreme ROIs (>500%): 0  (check for data errors)
5. Exits before investment date: 0  (should be 0)

